
这里调用huggingface上面的模型，对传入的图片进行拓展

In [42]:
from PIL import Image
from transformers import pipeline 

def img2text(url):    
    img_to_text = pipeline("image-to-text", model = "Salesforce/blip-image-captioning-base",device=0)
    # img = Image.open(url)  # 查看图片是否被打开
    # img.show()  # 展示图片
    text = img_to_text(url)[0]["generated_text"]    
    print(text)    
    return text


选择外接文心一言，这里直接传入

In [43]:
from zhipuai import ZhipuAI
url = "https://open.bigmodel.cn/api/paas/v4/chat/completions"
def text_extensions(text):
    client = ZhipuAI(api_key="b69f0e809e6601a4ca5c9d9407b8ce2b.Nwpl24uMiaVFZ6Im")  # 请填写您自己的APIKey
    
    response = client.chat.completions.create(
        model="glm-4-plus",  # 请填写您要调用的模型名称
        messages=[
            {"role": "user", "content": "I heard that you are a great writer, I will give you a sentence and expand the short story by 300 words"},
            {"role": "assistant", "content": "Of course, please provide your sentences"},
            {"role": "user", "content": text},
        ],
    )
    ex_text = response.choices[0].message.content
    print(ex_text)
    return ex_text

In [50]:
# if __name__ == 'main':
#     # #因为测试集只有七张，暂时这样做测试
#     # for i in range(7):
#     #     img_path = f"../img/img{i}.jpg"
#     #     text_extensions(img2text(img_path))

img_path = f"../img/img2.jpg"
ex_text = text_extensions(img2text(img_path))

two boys playing in the water
Under the golden glow of a late afternoon sun, two boys splashed and laughed in the gentle ripples of the lake. The water, cool and clear, mirrored the vibrant hues of the sky above, where wisps of clouds drifted lazily. The younger boy, Jamie, with hair the color of autumn leaves, shrieked with delight as he kicked up a spray of droplets, each one catching the light like tiny prisms. His older brother, Alex, tall and lanky with a mischievous grin, dove beneath the surface, emerging with a triumphant splash that sent ripples cascading towards the shore.

The lake was their sanctuary, a place where the worries of the world seemed to dissolve into the tranquil blue. They had spent countless summer days here, building castles of sand, racing each other to the floating dock, and inventing games that required nothing but their boundless imaginations. Today, they were pirates on a quest for hidden treasure, their ship a weathered old rowboat tethered to a crooke

In [51]:
print(ex_text)

Under the golden glow of a late afternoon sun, two boys splashed and laughed in the gentle ripples of the lake. The water, cool and clear, mirrored the vibrant hues of the sky above, where wisps of clouds drifted lazily. The younger boy, Jamie, with hair the color of autumn leaves, shrieked with delight as he kicked up a spray of droplets, each one catching the light like tiny prisms. His older brother, Alex, tall and lanky with a mischievous grin, dove beneath the surface, emerging with a triumphant splash that sent ripples cascading towards the shore.

The lake was their sanctuary, a place where the worries of the world seemed to dissolve into the tranquil blue. They had spent countless summer days here, building castles of sand, racing each other to the floating dock, and inventing games that required nothing but their boundless imaginations. Today, they were pirates on a quest for hidden treasure, their ship a weathered old rowboat tethered to a crooked wooden post.

Jamie clutched

下面调用翻译api，市面上可以用的随便，本来用的google，因为没充钱，现在改用百度

In [52]:
import requests
import random
import json
from hashlib import md5
 
# 设置你的appid/appkey
# appid = 'YOUR APP ID'
# appkey = 'YOU APP KEY'
appid = '20241112002201004'
appkey = 'rRqX2uzmWqLqplMV_M6c'

# 选择文本转换类型，`https://api.fanyi.baidu.com/doc/21`
from_lang = 'en'
to_lang =  'zh'
 
endpoint = 'http://api.fanyi.baidu.com'
path = '/api/trans/vip/translate'
url = endpoint + path
 
#在这里传入
query = ex_text
 
# Generate salt and sign
def make_md5(s, encoding='utf-8'):
    return md5(s.encode(encoding)).hexdigest()
 
def baidu_api(query,from_lang,to_lang):
    salt = random.randint(32768, 65536)
    sign = make_md5(appid + query + str(salt) + appkey)
 
    # 建立request
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    payload = {'appid': appid, 'q': query, 'from': from_lang, 'to': to_lang, 'salt': salt, 'sign': sign}
 
    # Send request
    r = requests.post(url, params=payload, headers=headers)
    result = r.json()
 
    # Show response
    #print(json.dumps(result, indent=4, ensure_ascii=False))
    return result["trans_result"][0]['dst']
 
print(baidu_api(query,from_lang, to_lang))

在傍晚金色的阳光下，两个男孩在湖面的涟漪中嬉戏大笑。凉爽而清澈的水，映照出上方天空的鲜艳色调，云朵懒洋洋地飘过。小男孩杰米的头发是秋叶的颜色，当他扬起一团水滴时，他高兴地尖叫起来，每一滴水滴都像小棱镜一样捕捉到光线。他的哥哥亚历克斯，又高又瘦，带着淘气的笑容，潜入水面，露出胜利的水花，激起阵阵涟漪，直冲岸边。


下面我们来实现文本转语音，也为此次课设最重要部分，在本地自己构建模型，然后进行训练,代码见extext_to_speech